# 01 Chunk With BGE-M3

Generate fixed, parent-child, and BGE-M3 semantic chunks. This writes artifacts only.

In [ ]:
import os, subprocess, sys
from pathlib import Path

PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', '/kaggle/input/tuvi-battu-graphrag/tuvi-battu-graphrag'))
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
DATASET_DIR = PROJECT_DIR / 'benchmark' / 'tuvi_golden_dataset'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
REPORTS = OUTPUT_ROOT / 'reports'
SOURCES = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
INPUTS = [str(DATASET_DIR / 'corpus' / source) for source in SOURCES]
STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
for strategy in STRATEGIES:
    cmd = [
        sys.executable, '-B', str(PROJECT_DIR / 'scripts' / 'chunk_text.py'),
        '--input', *INPUTS,
        '--chunking-strategy', strategy,
        '--output', str(OUTPUT_ROOT / 'chunks' / strategy),
        '--summary-output', str(REPORTS / f'{strategy}_chunk_summary.json'),
    ]
    if strategy == 'chunk_semantic_embedding_bge_m3':
        cmd += [
            '--semantic-report-output', str(REPORTS / f'{strategy}_semantic_similarity_report.json'),
            '--embedding-backend', 'local',
            '--local-embedding-model', 'BAAI/bge-m3',
            '--local-embedding-batch-size', '16',
        ]
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
